# 02 — Modélisation avec PyCaret (AutoML)Pipeline de modélisation automatisé :1. Chargement des données2. Configuration de l'environnement PyCaret3. Comparaison automatique des modèles4. Sélection du meilleur modèle5. Évaluation et interprétation> PyCaret est une librairie d'AutoML qui automatise tout le workflow de ML.> Une seule commande suffit pour entraîner et comparer 15+ modèles.

In [ ]:
# Importsimport pandas as pdimport numpy as npimport syssys.path.insert(0, '..')from src.data_loader import load_bank, load_bank_additionalfrom src.modeling import setup_pycaret, compare_all_modelsfrom src.evaluation import get_metrics, print_classification_report, compare_modelsfrom src.utils import set_seed# Imports PyCaret pour les visualisationsfrom pycaret.classification import plot_model, predict_model, finalize_model, save_modelset_seed(42)%matplotlib inline

## 1. Chargement des données

In [ ]:
# On charge le dataset Bank (UCI original)# PyCaret accepte le DataFrame complet (features + cible) :#   - La colonne 'y' sera la cible#   - La colonne 'duration' sera automatiquement gérée (on la garde pour l'EDA)#     mais on pourrait la supprimer avant si on veut un modèle plus réalistedf = load_bank(full=True)# Suppression de 'duration' pour un modèle réaliste# (cette colonne n'est pas connue avant la fin de l'appel)df = df.drop(columns=['duration'])print(f"Shape: {df.shape}")print(f"Colonnes: {list(df.columns)}")print(f"\nDistribution de la cible :")print(df['y'].value_counts())print(f"\nTaux de 'yes' : {df['y'].value_counts(normalize=True)['yes']:.1%}")

## 2. Configuration de l'environnement PyCaretLa fonction `setup()` initialise tout automatiquement :- Séparation train/test (80/20 avec stratification)- Encodage des variables catégorielles (one-hot encoding)- Standardisation des variables numériques- Validation croisée (10 folds)

In [ ]:
# Configuration PyCaret# - target='y' : la colonne à prédire# - train_size=0.8 : 80% entraînement, 20% test# - session_id=42 : seed pour la reproductibilité# - preprocess=True : active le prétraitement automatiquesetup_pycaret(df, target='y', train_size=0.8, session_id=42)

## 3. Comparaison automatique des modèles`compare_models()` entraîne et évalue 15+ modèles différents en une commande.Les modèles sont classés par Accuracy (comme demandé dans les instructions).

In [ ]:
# Lancer la comparaison automatique
# - sort='Accuracy' : classer par Accuracy (comme demandé)
# - n_select=5 : garder les 5 meilleurs modèles
# Retourne le meilleur modèle + le tableau de résultats
best_model, results = compare_all_models(sort='Accuracy', n_select=5)

# Le tableau de résultats s'affiche automatiquement avec les métriques :
# Accuracy, AUC, Recall, Precision, F1, Kappa, MCC, TT (temps d'entraînement)

## 4. Le meilleur modèlePyCaret a automatiquement sélectionné le meilleur modèle selon l'Accuracy.On va maintenant l'évaluer en détail.

In [ ]:
# Interprétation de l'Accuracy du meilleur modèle :print("📊 Interprétation de l'Accuracy :")print("-" * 50)print("L'Accuracy mesure le pourcentage de prédictions correctes.")print("MAIS dans ce projet, les classes sont déséquilibrées :")print(f"  - Non : {df['y'].value_counts(normalize=True)['no']:.1%}")print(f"  - Oui : {df['y'].value_counts(normalize=True)['yes']:.1%}")print()print("Un modèle qui prédit TOUJOURS 'non' aurait ~88% d'Accuracy...")print("et serait totalement inutile (0% des vrais 'oui' trouvés).")print()print("Il faut donc regarder AUSSI :")print("  - Recall : combien de vrais 'oui' sont trouvés ?")print("  - Precision : parmi les 'oui' prédits, combien sont corrects ?")print("  - ROC-AUC : le modèle distingue-t-il bien les deux classes ?")

## 5. Évaluation visuelle du meilleur modèle

In [ ]:
# Matrice de confusion : qui est bien classé, qui est confondu ?# Format :#                  | Prédit NON | Prédit OUI |#   Vrai NON       |     VN     |     FP     |#   Vrai OUI       |     FN     |     VP     |plot_model(best_model, plot='confusion_matrix')

In [ ]:
# Courbe ROC : le modèle distingue-t-il bien les 'oui' des 'non' ?# AUC = 1.0 → parfait | AUC = 0.5 → hasard | AUC < 0.5 → pire que le hasardplot_model(best_model, plot='auc')

In [ ]:
# Importance des features : qu'est-ce qui pèse le plus dans la décision ?plot_model(best_model, plot='feature')

## 6. Prédictions sur l'ensemble de test

In [ ]:
# Faire des prédictions sur l'ensemble de test# predict_model() utilise automatiquement les données de testpredictions = predict_model(best_model)# Extraire les vraies valeurs et les prédictionsy_test = predictions['y']y_pred = predictions['prediction_label']y_proba = predictions['prediction_score']# Afficher le rapport de classification détailléprint_classification_report(y_test, y_pred)

## 7. Sauvegarde du modèle

In [ ]:
# Finaliser le modèle (ré-entraîner sur TOUTES les données)# et le sauvegarder pour utilisation futurefinal_model = finalize_model(best_model)save_model(final_model, '../models/best_model_pycaret')print("✅ Modèle sauvegardé dans models/best_model_pycaret.pkl")

## 8. Conclusions de la modélisation- PyCaret a automatiquement comparé 15+ modèles de classification- Le meilleur modèle selon l'Accuracy est affiché ci-dessus- L'Accuracy seule n'est pas suffisante dans ce projet à cause du déséquilibre des classes- Les métriques complémentaires (Recall, ROC-AUC) confirment la qualité réelle du modèle- La matrice de confusion montre combien de 'oui' ont été correctement identifiés- Les features les plus importantes guident la stratégie marketing